In [1]:
!pip install mysql-connector-python pandas

In [2]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host="localhost",
    user="root",              # your MySQL username
    password="Vinod007", # your actual MySQL password
    database="e-commerce_project"      # the database name you created
)

df = pd.read_sql("SELECT * FROM customers LIMIT 5;", conn)
df

C:\Users\Admin\AppData\Local\Temp\ipykernel_19048\1028085673.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM customers LIMIT 5;", conn)


,customer_id,customer_name,email,gender,age,city,state,signup_date,acquisition_channel
0,1,Anvi Konda,liamchaudry@example.net,Male,19,Kochi,Kerala,2024-10-23,Social Media
1,2,Pahal Balay,chandertejas@example.org,Male,26,Hyderabad,Telangana,2025-02-04,Social Media
2,3,Rushil Saini,saumyamall@example.org,Male,55,Kochi,Kerala,2024-06-06,Paid Ads
3,4,Pahal Oak,nachiket35@example.org,Male,31,Bengaluru,Karnataka,2025-02-23,Organic Search
4,5,Jagrati Padmanabhan,caleb78@example.org,Male,53,Lucknow,Uttar Pradesh,2023-04-06,Organic Search


In [3]:
query = """
SELECT o.customer_id, o.order_id, o.order_date
FROM orders o
WHERE o.order_status = 'Delivered'
"""
orders_df = pd.read_sql(query, conn)
orders_df.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_19048\3085222348.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders_df = pd.read_sql(query, conn)


,customer_id,order_id,order_date
0,932,1,2024-12-18
1,34,2,2024-11-19
2,571,3,2025-08-13
3,825,4,2024-11-19
4,344,5,2025-12-30


In [4]:
orders_df

,customer_id,order_id,order_date
0,932,1,2024-12-18
1,34,2,2024-11-19
2,571,3,2025-08-13
3,825,4,2024-11-19
4,344,5,2025-12-30
...,...,...,...
5961,1005,8994,2025-03-04
5962,243,8995,2025-05-31
5963,864,8996,2025-12-24
5964,652,8997,2024-08-13


In [5]:
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])

first_purchase = orders_df.groupby('customer_id')['order_date'].min().reset_index()
first_purchase.columns = ['customer_id', 'first_purchase_date']
first_purchase.head()

,customer_id,first_purchase_date
0,1,2025-01-11
1,2,2025-02-04
2,3,2024-07-01
3,4,2025-03-01
4,5,2025-01-09


In [6]:
orders_df = orders_df.merge(first_purchase, on='customer_id', how='left')
orders_df.head()

,customer_id,order_id,order_date,first_purchase_date
0,932,1,2024-12-18,2024-12-05
1,34,2,2024-11-19,2024-11-15
2,571,3,2025-08-13,2024-10-31
3,825,4,2024-11-19,2024-01-17
4,344,5,2025-12-30,2025-12-30


In [7]:
orders_df['cohort_month'] = orders_df['first_purchase_date'].dt.to_period('M')
orders_df['order_month'] = orders_df['order_date'].dt.to_period('M')

orders_df['month_index'] = (
    (orders_df['order_month'].dt.year - orders_df['cohort_month'].dt.year) * 12 +
    (orders_df['order_month'].dt.month - orders_df['cohort_month'].dt.month)
)

orders_df.head()

,customer_id,order_id,order_date,first_purchase_date,cohort_month,order_month,month_index
0,932,1,2024-12-18,2024-12-05,2024-12,2024-12,0
1,34,2,2024-11-19,2024-11-15,2024-11,2024-11,0
2,571,3,2025-08-13,2024-10-31,2024-10,2025-08,10
3,825,4,2024-11-19,2024-01-17,2024-01,2024-11,10
4,344,5,2025-12-30,2025-12-30,2025-12,2025-12,0


In [8]:
cohort_counts = orders_df.groupby(['cohort_month', 'month_index'])['customer_id'].nunique().reset_index()
cohort_counts.columns = ['cohort_month', 'month_index', 'num_customers']
cohort_counts.head(10)

,cohort_month,month_index,num_customers
0,2024-01,0,85
1,2024-01,1,16
2,2024-01,2,22
3,2024-01,3,16
4,2024-01,4,26
5,2024-01,5,19
6,2024-01,6,17
7,2024-01,7,27
8,2024-01,8,15
9,2024-01,9,34


In [9]:
cohort_sizes = cohort_counts[cohort_counts['month_index'] == 0][['cohort_month', 'num_customers']]
cohort_sizes.columns = ['cohort_month', 'cohort_size']

cohort_counts = cohort_counts.merge(cohort_sizes, on='cohort_month', how='left')
cohort_counts['retention_pct'] = round(cohort_counts['num_customers'] / cohort_counts['cohort_size'] * 100, 1)

cohort_counts.head(10)

,cohort_month,month_index,num_customers,cohort_size,retention_pct
0,2024-01,0,85,85,100.0
1,2024-01,1,16,85,18.8
2,2024-01,2,22,85,25.9
3,2024-01,3,16,85,18.8
4,2024-01,4,26,85,30.6
5,2024-01,5,19,85,22.4
6,2024-01,6,17,85,20.0
7,2024-01,7,27,85,31.8
8,2024-01,8,15,85,17.6
9,2024-01,9,34,85,40.0


In [10]:
retention_matrix = cohort_counts.pivot(index='cohort_month', columns='month_index', values='retention_pct')
retention_matrix

month_index,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
cohort_month,,,,,,,,,,,,,,,,,,,,,
2024-01,100.0,18.8,25.9,18.8,30.6,22.4,20.0,31.8,17.6,40.0,...,18.8,20.0,21.2,29.4,21.2,22.4,18.8,45.9,37.6,30.6
2024-02,100.0,18.5,15.4,18.5,21.5,13.8,16.9,16.9,43.1,29.2,...,15.4,23.1,21.5,23.1,15.4,23.1,40.0,30.8,23.1,NaN
2024-03,100.0,23.9,21.7,30.4,30.4,26.1,8.7,50.0,26.1,32.6,...,19.6,21.7,15.2,21.7,17.4,41.3,30.4,17.4,NaN,NaN
2024-04,100.0,26.8,24.4,26.8,22.0,22.0,39.0,29.3,14.6,26.8,...,12.2,24.4,19.5,14.6,43.9,29.3,17.1,NaN,NaN,NaN
2024-05,100.0,21.8,14.5,25.5,21.8,45.5,25.5,23.6,25.5,21.8,...,27.3,23.6,9.1,38.2,12.7,23.6,NaN,NaN,NaN,NaN
2024-06,100.0,29.2,22.9,18.8,43.8,22.9,20.8,29.2,22.9,14.6,...,22.9,20.8,37.5,45.8,20.8,NaN,NaN,NaN,NaN,NaN
2024-07,100.0,26.5,5.9,35.3,14.7,17.6,23.5,23.5,23.5,11.8,...,14.7,32.4,20.6,20.6,NaN,NaN,NaN,NaN,NaN,NaN
2024-08,100.0,18.8,29.2,27.1,25.0,18.8,12.5,18.8,12.5,16.7,...,37.5,18.8,16.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-09,100.0,38.0,26.0,18.0,22.0,10.0,12.0,24.0,20.0,10.0,...,22.0,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
retention_matrix.to_csv("cohort_retention.csv")

In [12]:
import os
print(os.getcwd())

C:\Users\Admin


In [13]:
import pandas as pd
check = pd.read_csv("cohort_retention.csv")
check.head()

,cohort_month,0,1,2,3,4,5,6,7,8,...,14,15,16,17,18,19,20,21,22,23
0,2024-01,100.0,18.8,25.9,18.8,30.6,22.4,20.0,31.8,17.6,...,18.8,20.0,21.2,29.4,21.2,22.4,18.8,45.9,37.6,30.6
1,2024-02,100.0,18.5,15.4,18.5,21.5,13.8,16.9,16.9,43.1,...,15.4,23.1,21.5,23.1,15.4,23.1,40.0,30.8,23.1,NaN
2,2024-03,100.0,23.9,21.7,30.4,30.4,26.1,8.7,50.0,26.1,...,19.6,21.7,15.2,21.7,17.4,41.3,30.4,17.4,NaN,NaN
3,2024-04,100.0,26.8,24.4,26.8,22.0,22.0,39.0,29.3,14.6,...,12.2,24.4,19.5,14.6,43.9,29.3,17.1,NaN,NaN,NaN
4,2024-05,100.0,21.8,14.5,25.5,21.8,45.5,25.5,23.6,25.5,...,27.3,23.6,9.1,38.2,12.7,23.6,NaN,NaN,NaN,NaN


In [14]:
rfm_query = """
WITH cte_4 AS (
    SELECT 
        o.customer_id,
        MAX(o.order_date) AS last_order_date,
        COUNT(DISTINCT o.order_id) AS frequency,
        ROUND(SUM(total_amount), 2) AS monetary,
        (SELECT MAX(order_date) FROM orders WHERE order_status = 'Delivered') AS snapshot_date
    FROM orders o 
    JOIN order_items od ON o.order_id = od.order_id
    WHERE o.order_status = 'Delivered'
    GROUP BY o.customer_id
),
cte_5 AS (
    SELECT
        customer_id,
        frequency,
        monetary,
        DATEDIFF(snapshot_date, last_order_date) AS recency
    FROM cte_4
),
rfm_scored AS (
    SELECT
        customer_id,
        recency,
        frequency,
        monetary,
        NTILE(4) OVER (ORDER BY recency DESC) AS r_score,
        NTILE(4) OVER (ORDER BY frequency ASC) AS f_score,
        NTILE(4) OVER (ORDER BY monetary ASC) AS m_score
    FROM cte_5
),
rfm_final as(
SELECT
    customer_id,
    recency,
    frequency,
    monetary,
    r_score, f_score, m_score,
    (r_score + f_score + m_score) AS rfm_score
FROM rfm_scored
ORDER BY rfm_score DESC
)

SELECT
    customer_id,
    recency,
    frequency,
    monetary,
    r_score, f_score, m_score,
    (r_score + f_score + m_score) AS rfm_score,
    CASE
        WHEN rfm_score >= 10 THEN 'Champions'
        WHEN rfm_score >= 8  THEN 'Loyal Customers'
        WHEN rfm_score >= 6  THEN 'Potential Loyalists'
        WHEN rfm_score >= 4  THEN 'Needs Attention'
        ELSE 'At Risk / Churned'
    END AS segment
FROM rfm_final
ORDER BY rfm_score DESC;

"""
rfm_df = pd.read_sql(rfm_query, conn)
rfm_df.to_csv("customer_rfm.csv", index=False)

C:\Users\Admin\AppData\Local\Temp\ipykernel_19048\2997518569.py:63: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rfm_df = pd.read_sql(rfm_query, conn)


In [15]:
rfm_df

,customer_id,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,segment
0,337,5,11,304090.56,4,4,4,12,Champions
1,612,8,8,310793.44,4,4,4,12,Champions
2,507,26,10,317775.62,4,4,4,12,Champions
3,113,3,13,322027.55,4,4,4,12,Champions
4,514,10,14,324219.15,4,4,4,12,Champions
...,...,...,...,...,...,...,...,...,...
1040,132,404,1,40518.72,1,1,1,3,At Risk / Churned
1041,180,546,1,41813.66,1,1,1,3,At Risk / Churned
1042,107,426,1,41982.47,1,1,1,3,At Risk / Churned
1043,1127,200,1,43896.79,1,1,1,3,At Risk / Churned


In [16]:
fact_query = """
SELECT 
    oi.order_item_id,
    o.order_id,
    o.customer_id,
    o.order_date,
    o.order_status,
    o.payment_method,
    oi.product_id,
    oi.quantity,
    oi.unit_price,
    oi.discount_pct,
    oi.total_amount,
    p.category,
    p.sub_category,
    p.brand,
    p.cost_price,
    c.city,
    c.state
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN products p ON oi.product_id = p.product_id
JOIN customers c ON o.customer_id = c.customer_id
"""

fact_df = pd.read_sql(fact_query, conn)
fact_df.to_csv("fact_sales.csv", index=False)
fact_df.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_19048\2645478728.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fact_df = pd.read_sql(fact_query, conn)


,order_item_id,order_id,customer_id,order_date,order_status,payment_method,product_id,quantity,unit_price,discount_pct,total_amount,category,sub_category,brand,cost_price,city,state
0,54,33,13,2024-03-02,Delivered,Net Banking,98,1,1006.62,20,805.30,Beauty & Personal Care,Fragrances,WOW,849.01,Nagpur,Maharashtra
1,55,33,13,2024-03-02,Delivered,Net Banking,81,1,658.10,10,592.29,Beauty & Personal Care,Haircare,Mamaearth,524.14,Nagpur,Maharashtra
2,189,114,6,2025-11-26,Delivered,Debit Card,22,2,66401.24,5,126162.36,Electronics,Cameras,Sony,41635.34,Kochi,Kerala
3,470,277,13,2024-07-21,Returned,Cash on Delivery,148,1,2019.62,10,1817.66,Books & Stationery,Non-Fiction,Scholastic,1496.73,Nagpur,Maharashtra
4,599,344,12,2024-07-26,Delivered,UPI,46,1,2338.81,0,2338.81,Fashion,Men's Clothing,Puma,1962.33,Surat,Gujarat
